# Bài 10 · BatchNorm, LayerNorm & RMSNorm: tự viết ba phép chuẩn hóa và thử chúng trên mạng MNIST sâu 10–30 lớp

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/10-chuan-hoa.ipynb)

Notebook đi kèm [Bài 10 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/chuan-hoa/) (demo tương tác nằm ở trang bài học).
Ở Bài 09, mạng sâu 784 → L×64 → 10 khó huấn luyện vì tín hiệu và gradient co lại hoặc phình ra qua từng lớp. Ở đây ta
chèn một bước **chuẩn hóa** vào mỗi lớp ẩn (Linear → chuẩn hóa → ReLU), tự viết bằng numpy BatchNorm, LayerNorm và
RMSNorm cả chiều xuôi lẫn chiều ngược, rồi đo trên MNIST xem chúng giúp mạng sâu đến đâu, và chỗ nào thì không.

**Bạn sẽ làm:**
1. Viết BatchNorm, LayerNorm, RMSNorm (xuôi và ngược) và kiểm tra gradient bằng sai phân hữu hạn.
2. Chèn chúng vào mạng sâu, đo xem phân bố đầu vào của một lớp ẩn trôi đi thế nào khi huấn luyện, và thử cố tình
   làm nó trôi.
3. Áp ba cách lên kích hoạt thật của một mạng đã học: chuẩn hóa theo cột (theo lô) hay theo hàng (theo mẫu), và
   γ, β để làm gì.
4. So sánh BatchNorm lúc huấn luyện (thống kê của lô) với lúc suy luận (trung bình trượt).
5. Huấn luyện mạng 10–30 lớp với từng cách, đọc kết quả quét đầy đủ, và tìm lý do BatchNorm không cứu được mạng
   30 lớp.

**Cần biết trước:** [Bài 09 · Gradient biến mất & khởi tạo](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/vanishing-gradient/)
(cùng mạng sâu, khởi tạo He), [Bài 06 · Lan truyền ngược](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/lan-truyen-nguoc/)
(chiều ngược, dùng lại ở mục 5) và [Bài 05 · Gradient descent](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/gradient-descent/)
(SGD, tốc độ học $\eta$). **Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải MNIST (~11 MB) lần đầu.

Mọi con số trong notebook đều do chính các ô bên dưới tính ra, và trùng với số trong demo. Riêng kết quả quét đầy đủ
(216 lần huấn luyện) được nạp từ đúng tệp mà demo dùng, vì chạy lại mất vài phút; notebook chạy lại vài ô của nó và
kiểm tra rằng ra cùng con số.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi.

In [ ]:
import gzip
import hashlib
import json
import math
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, ScalarFormatter

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (hình dạng, nét gạch, nhãn)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("cam_xam_xanh", [ORANGE, "#f2f2f2", BLUE])  # âm → 0 → dương
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# Mỗi cách chuẩn hóa một màu, một dạng điểm và một kiểu nét, như trong demo
NORMS = ["none", "bn", "ln", "rms"]
LABEL = {"none": "Không chuẩn hóa", "bn": "BatchNorm", "ln": "LayerNorm", "rms": "RMSNorm"}
STYLE = {
    "none": {"color": GREY, "marker": "o", "mfc": "white", "ls": (0, (2, 3))},
    "bn": {"color": BLUE, "marker": "o", "mfc": BLUE, "ls": "-"},
    "ln": {"color": ORANGE, "marker": "s", "mfc": ORANGE, "ls": (0, (6, 3))},
    "rms": {"color": GREEN, "marker": "^", "mfc": GREEN, "ls": (0, (1.5, 3))},
}


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38."""
    x = round(float(x), digits) + 0.0  # + 0.0: không in “−0”
    s = f"{x:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


def vn_eta(e):
    """η có một chữ số có nghĩa: 0,3 · 0,03 · 1."""
    return vn(e, max(0, -math.floor(math.log10(e) + 1e-9)))


SUP = str.maketrans("0123456789-", "⁰¹²³⁴⁵⁶⁷⁸⁹⁻")


def sci(x, digits=1):
    """Số rất nhỏ: 2,7·10⁻⁶."""
    if x == 0:
        return "0"
    m, e = f"{x:.{digits}e}".split("e")
    return f"{m.replace('.', ',')}·10{str(int(e)).translate(SUP)}"


def vn_tick(v, _pos=None):
    """Nhãn vạch trên trục: 0,5 · −2 · 1.000 (bỏ số 0 thừa ở cuối)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def vn_axes(fig):
    """Đổi nhãn số của mọi trục sang kiểu Việt Nam (nhãn chữ giữ nguyên). Trục log lấy vạch 1–2–5, hoặc lũy thừa
    của 10 nếu trục rất dài."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() == "log":
                lo, hi = sorted(lim)
                mult = (1, 2, 5) if hi / lo < 300 else (1,)
                ticks = [m * 10.0**k for k in range(-4, 5) for m in mult if lo <= m * 10.0**k <= hi]
                axis.set_major_locator(FixedLocator(ticks))
                axis.set_minor_formatter(NullFormatter())
                axis.set_major_formatter(FuncFormatter(vn_tick))
            elif isinstance(axis.get_major_formatter(), ScalarFormatter):  # trục số tuyến tính, không phải nhãn chữ
                axis.set_major_formatter(FuncFormatter(vn_tick))


print(vn(13007, 0), vn(0.1, 1), vn(-4.3805, 4), pct(0.953213), vn_eta(0.03), sci(2.7e-6), vn_tick(1000.0))

## 2 · Dữ liệu

MNIST (LeCun và cộng sự, 1998): 60.000 ảnh huấn luyện và 10.000 ảnh kiểm tra, chữ số viết tay xám 28×28. Ô dưới tải
4 tệp gốc và kiểm tra mã MD5, nên ai chạy cũng có đúng cùng một dữ liệu. Giống demo, ta chia ba phần:
- **huấn luyện:** 55.000 ảnh đầu, để tính gradient;
- **kiểm định:** 5.000 ảnh cuối của tệp huấn luyện, để so sánh và chọn $\eta$;
- **kiểm tra:** 10.000 ảnh của tệp kiểm tra, chỉ để báo kết quả cuối.

Ô cũng nạp tệp kết quả quét mà demo dùng (`fetch_json`, có kiểm tra MD5).

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # mã MD5 chuẩn của MNIST
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"


def load(kind):
    """Định dạng IDX: 16 byte đầu của tệp ảnh (8 byte của tệp nhãn) là phần mô tả, còn lại là các byte 0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


# Kết quả của một lần huấn luyện dài (cùng tệp với demo trên web). build.py thay COURSE_DATA_URL bằng thư mục data/
# của repo notebook công khai.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


fetch()
sweep = fetch_json("mnist-norm.json", "d0fc66c752fdbd26987077feabbfc1d0")  # kết quả quét của demo

In [ ]:
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
X_all, y_all = train_img.reshape(-1, 784) / 255.0, train_lab.astype(np.int64)
Xtr, ytr = X_all[:55000], y_all[:55000]  # huấn luyện
Xval, yval = X_all[55000:], y_all[55000:]  # kiểm định
Xte, yte = test_img.reshape(-1, 784) / 255.0, test_lab.astype(np.int64)  # kiểm tra: chỉ để báo cáo
print(f"huấn luyện {vn(len(ytr), 0)} · kiểm định {vn(len(yval), 0)} · kiểm tra {vn(len(yte), 0)} ảnh")
n_runs = sum(len(r["seeds"]) for r in sweep["runs"])
print(
    f"tệp quét: {len(sweep['norms'])} cách chuẩn hóa × {len(sweep['depths'])} độ sâu × {len(sweep['etas'])} giá trị η "
    f"× {len(sweep['seeds'])} seed = {n_runs} lần huấn luyện 1 epoch"
)
assert (len(ytr), len(yval), len(yte)) == (sweep["nTrain"], sweep["nVal"], sweep["nTest"])

Tám ảnh kiểm tra đầu tiên. Ở mục 7 ta sẽ nhìn kích hoạt thật của mạng trên đúng tám ảnh này, như demo:

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(8, 1.4))
for i, ax in enumerate(axes):
    ax.imshow(test_img[i], cmap="gray_r")
    ax.set_title(f"ảnh {i + 1}: số {test_lab[i]}", fontsize=9)
    ax.axis("off")
plt.show()

## 3 · Mạng sâu của Bài 09, và đầu vào của một lớp ẩn

Ta dùng lại đúng mạng của Bài 09: 784 → $L$ lớp ẩn × 64 nơ-ron → 10, ReLU, khởi tạo He ($W \sim \mathcal N(0, 2/n_{in})$),
độ lệch bằng 0. Giống demo, các số ngẫu nhiên được sinh bằng `mulberry32` (bộ sinh số của JavaScript mà demo dùng) rồi
biến đổi Box–Muller thành phân phối chuẩn, nên seed 1 cho đúng mạng mà demo Bài 09 vẽ, và độ sâu $L$ chỉ là “lấy
$L$ lớp đầu”. Ô dưới tạo mạng 10 lớp và đếm tham số.

In [ ]:
WIDTH, BATCH, N_TRAIN = 64, 64, 55000
EPS, MOM = 1e-5, 0.1  # ε của phép chuẩn hóa; hệ số của trung bình trượt (mục 8)


def mulberry32(seed, n):
    """n số đầu tiên của bộ sinh mulberry32(seed) trong JavaScript, chia cho 2³² để thành số thực trong [0, 1)."""
    a = ((seed + np.arange(1, n + 1, dtype=np.uint64) * 0x6D2B79F5) & 0xFFFFFFFF).astype(np.uint32)
    t = (a ^ (a >> 15)) * (a | 1)
    t = (t + (t ^ (t >> 7)) * (t | 61)) ^ t
    return (t ^ (t >> 14)).astype(np.float64) / 2**32


def init(norm, depth, seed=1):
    """Mạng 784 → depth×64 → 10 với cách chuẩn hóa `norm` ∈ {none, bn, ln, rms}; γ = 1, β = 0 lúc đầu."""
    shapes = [(WIDTH, 10), (784, WIDTH)] + [(WIDTH, WIDTH)] * (depth - 1)  # thứ tự rút số giống demo
    sizes = [r * c for r, c in shapes]
    u = mulberry32(seed, 2 * sum(sizes))
    z = np.sqrt(-2 * np.log(1 - u[0::2])) * np.cos(2 * np.pi * u[1::2])  # Box–Muller: mỗi cặp số đều → một số chuẩn
    e = [part.reshape(s) for part, s in zip(np.split(z, np.cumsum(sizes)[:-1]), shapes)]
    W = [np.sqrt(2 / r) * ei for (r, _), ei in zip(shapes[1:], e[1:])] + [np.sqrt(2 / WIDTH) * e[0]]  # He
    return {
        "norm": norm,
        "depth": depth,
        "W": W,
        "b": [np.zeros(w.shape[1]) for w in W],
        "g": [np.ones(WIDTH) for _ in range(depth)],  # γ của mỗi lớp ẩn
        "be": [np.zeros(WIDTH) for _ in range(depth)],  # β (RMSNorm không dùng)
        "rm": [np.zeros(WIDTH) for _ in range(depth)],  # trung bình trượt của BatchNorm (mục 8)
        "rv": [np.ones(WIDTH) for _ in range(depth)],  # phương sai trượt của BatchNorm
    }


P10 = init("none", 10)
n_par = sum(w.size + b.size for w, b in zip(P10["W"], P10["b"]))
print(f"mạng 10 lớp ẩn: {len(P10['W'])} ma trận W, {vn(n_par, 0)} tham số (chưa kể γ, β)")
print("W₁:", P10["W"][0].shape, " W₂…W₁₀:", P10["W"][1].shape, " W ra:", P10["W"][-1].shape)
assert n_par == 784 * 64 + 64 + 9 * (64 * 64 + 64) + 64 * 10 + 10

Mỗi lớp ẩn tính $a = hW + b$ rồi $h \leftarrow \mathrm{ReLU}(a)$. Ta gọi $a$ là **đầu vào của lớp chuẩn hóa**: bài
này chèn một bước chuẩn hóa vào giữa, thành Linear → chuẩn hóa → ReLU. Ô dưới chạy 8 ảnh kiểm tra đầu tiên qua 4 lớp
đầu của mạng chưa huấn luyện, rồi in $a$ của lớp thứ 5 cho 6 trong 64 nơ-ron. Mỗi hàng là một ảnh, mỗi cột là một
nơ-ron:

In [ ]:
h = Xte[:8]
for W in P10["W"][:4]:
    h = np.maximum(0, h @ W)  # 4 lớp ẩn đầu (b = 0 lúc khởi tạo)
a5 = h @ P10["W"][4]  # đầu vào của lớp chuẩn hóa thứ 5: 8 ảnh × 64 nơ-ron
print("a của lớp 5 (8 ảnh × 6 nơ-ron đầu):")
for i, row in enumerate(a5[:, :6]):
    print(f"  ảnh {i + 1} (số {yte[i]}): " + "  ".join(f"{vn(v, 2):>6}" for v in row))
mu_col, sd_col = a5.mean(0), a5.std(0)
print(f"trung bình theo cột (64 nơ-ron) từ {vn(mu_col.min())} đến {vn(mu_col.max())}; ", end="")
print(f"độ lệch chuẩn từ {vn(sd_col.min())} đến {vn(sd_col.max())}")

Mỗi nơ-ron có thang đo riêng: cột đầu âm với cả 8 ảnh, cột thứ hai dương với cả 8 ảnh, và độ lệch chuẩn giữa các
cột chênh nhau nhiều lần. Lớp sau nhận đúng những con số này. Khi huấn luyện, các lớp trước thay đổi, nên thang đo
của từng cột cũng đổi theo (mục 6 đo điều này).

## 4 · Trừ trung bình, chia độ lệch chuẩn: ba cách chọn trục

Chuẩn hóa đưa một nhóm số về trung bình 0, phương sai 1, rồi cho mạng tự học lại thang đo $\gamma$ và độ dời $\beta$:

$$\hat x = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \qquad y = \gamma\,\hat x + \beta$$

($\sigma^2$ ở đây là phương sai, không phải hàm sigmoid; $\epsilon = 10^{-5}$ tránh chia cho 0.) Ba cách chỉ khác
nhau ở chỗ **lấy $\mu, \sigma^2$ trên nhóm số nào** của ma trận $a$ (hàng = ảnh, cột = nơ-ron):

- **BatchNorm** (Ioffe & Szegedy, 2015): theo **cột**, tức mỗi nơ-ron trên các ảnh của lô.
- **LayerNorm** (Ba et al., 2016): theo **hàng**, tức mỗi ảnh trên 64 nơ-ron của nó, không cần ảnh nào khác.
- **RMSNorm** (Zhang & Sennrich, 2019): cũng theo hàng, nhưng **không trừ trung bình** và không có $\beta$; chỉ chia
  cho căn của trung bình bình phương, $\mathrm{RMS}(x) = \sqrt{\tfrac1d \sum_i x_i^2 + \epsilon}$.

Ta làm tay cột đầu tiên bằng một vòng lặp, rồi viết bản vectơ hóa:

In [ ]:
col = a5[:, 0]
mu = sum(col) / len(col)
var = sum((v - mu) ** 2 for v in col) / len(col)  # phương sai “chia cho n”, như trong bài báo
xhat_loop = [(v - mu) / math.sqrt(var + EPS) for v in col]
print(f"cột 1: μ = {vn(mu, 4)}, σ² = {vn(var, 4)} → x̂ =", " ".join(vn(v, 2) for v in xhat_loop))
print(f"sau chuẩn hóa: trung bình {vn(np.mean(xhat_loop), 4)}, độ lệch chuẩn {vn(np.std(xhat_loop), 4)}")


def norm_forward(a, kind, gamma, beta):
    """BatchNorm (kind='bn', theo cột), LayerNorm ('ln', theo hàng) hoặc RMSNorm ('rms'). Trả về y và cache cho chiều ngược."""
    if kind == "rms":
        r = np.sqrt((a * a).mean(1, keepdims=True) + EPS)  # RMS của từng hàng
        xhat = a / r
        return gamma * xhat, (xhat, r)  # RMSNorm chỉ có γ
    axis = 0 if kind == "bn" else 1  # bn: trung bình trên các ảnh; ln: trên các nơ-ron của một ảnh
    inv = 1 / np.sqrt(a.var(axis, keepdims=True) + EPS)
    xhat = (a - a.mean(axis, keepdims=True)) * inv
    return gamma * xhat + beta, (xhat, inv)


one, zero = np.ones(WIDTH), np.zeros(WIDTH)
y_bn, y_ln, y_rms = (norm_forward(a5, k, one, zero)[0] for k in ("bn", "ln", "rms"))
assert np.allclose(y_bn[:, 0], xhat_loop)  # bản vectơ hóa = vòng lặp
for name, y in [("BatchNorm", y_bn), ("LayerNorm", y_ln), ("RMSNorm", y_rms)]:
    c_mu, r_mu, r_sd = np.abs(y.mean(0)).max(), np.abs(y.mean(1)).max(), y.std(1)
    r_rms = np.sqrt((y * y).mean(1))
    print(
        f"{name:>9}: |trung bình cột| ≤ {vn(c_mu, 3)} · |trung bình hàng| ≤ {vn(r_mu, 3)} · "
        f"độ lệch chuẩn hàng {vn(r_sd.min(), 3)}–{vn(r_sd.max(), 3)} · RMS hàng {vn(r_rms.min(), 3)}–{vn(r_rms.max(), 3)}"
    )

Vòng lặp và bản vectơ hóa cho cùng một cột. Sau chuẩn hóa, cột đó có trung bình 0 và độ lệch chuẩn 0,9996 chứ không
đúng 1: phương sai của nó chỉ 0,0118, nên $\epsilon = 10^{-5}$ bắt đầu có ảnh hưởng. Ba dòng cuối đọc như sau:

- **BatchNorm** đưa mọi **cột** về trung bình 0, nhưng từng hàng (từng ảnh) vẫn lệch.
- **LayerNorm** đưa mọi **hàng** về trung bình 0 và độ lệch chuẩn 1; các cột thì không.
- **RMSNorm** chỉ làm cho mỗi hàng có RMS = 1; trung bình của hàng vẫn khác 0. Bỏ bước trừ trung bình, nó rẻ hơn
  LayerNorm một phép tính trên mỗi hàng.

## 5 · Chiều ngược, và kiểm tra bằng sai phân hữu hạn

Để huấn luyện, ta cần $\partial L/\partial a$ khi biết $\partial L/\partial y$. Cái khó là $a$ ảnh hưởng tới $\hat x$
theo ba đường: trực tiếp, qua $\mu$ và qua $\sigma$. Đặt $g = \gamma \odot \partial L/\partial y$ (gradient theo
$\hat x$), và gạch ngang trên đầu là trung bình trên nhóm được chuẩn hóa (cột với BatchNorm, hàng với LayerNorm).
Lấy đạo hàm cả ba đường rồi rút gọn, ta được:

$$\frac{\partial L}{\partial a} = \frac{1}{\sqrt{\sigma^2+\epsilon}}\Big(g - \overline{g} - \hat x \odot \overline{g \odot \hat x}\Big)
\qquad \frac{\partial L}{\partial \gamma} = \sum_{\text{ảnh}} \frac{\partial L}{\partial y} \odot \hat x
\qquad \frac{\partial L}{\partial \beta} = \sum_{\text{ảnh}} \frac{\partial L}{\partial y}$$

Số hạng $\overline g$ đến từ $\mu$, số hạng cuối từ $\sigma$. RMSNorm không trừ $\mu$, nên chỉ mất $\overline g$ và
thay $\sigma$ bằng RMS. Công thức này đúng cả khi có $\epsilon$. Ô dưới viết nó thành code, rồi so với gradient số
$\big(L(a+h) - L(a-h)\big)/2h$ trên một ma trận ngẫu nhiên nhỏ, với $L = \sum R \odot y$ cho một ma trận $R$ bất kỳ.

In [ ]:
def norm_backward(dy, kind, cache, gamma):
    """Chiều ngược của norm_forward: từ dL/dy ra dL/da, dL/dγ, dL/dβ."""
    xhat, s = cache  # s = 1/σ (BatchNorm, LayerNorm) hoặc RMS (RMSNorm)
    dgamma, dbeta = (dy * xhat).sum(0), dy.sum(0)  # γ, β thuộc về từng nơ-ron: cộng trên các ảnh
    g = dy * gamma  # dL/dx̂
    if kind == "rms":  # không có μ: chỉ còn đường trực tiếp và đường qua RMS
        return (g - xhat * (g * xhat).mean(1, keepdims=True)) / s, dgamma, dbeta
    axis = 0 if kind == "bn" else 1
    m = g.shape[axis]  # số giá trị trong mỗi nhóm: m·ḡ = tổng của g
    da = s / m * (m * g - g.sum(axis, keepdims=True) - xhat * (g * xhat).sum(axis, keepdims=True))
    return da, dgamma, dbeta


def numgrad(f, x, h=1e-6):
    """Gradient số của f() theo mảng x (sửa x tại chỗ rồi trả lại như cũ)."""
    out = np.zeros_like(x)
    for i in np.ndindex(x.shape):
        old = x[i]
        x[i] = old + h
        up = f()
        x[i] = old - h
        down = f()
        x[i] = old
        out[i] = (up - down) / (2 * h)
    return out


def rel_err(u, v):
    return float(np.abs(u - v).max() / (np.abs(u).max() + np.abs(v).max()))


rng = np.random.default_rng(0)
a_t, R = rng.normal(1.0, 2.0, (6, 5)), rng.normal(size=(6, 5))  # 6 “ảnh” × 5 “nơ-ron”
g_t, b_t = rng.normal(1.0, 0.3, 5), rng.normal(0.0, 0.3, 5)
worst = 0.0
for kind in ("bn", "ln", "rms"):
    da, dg, dbe = norm_backward(R, kind, norm_forward(a_t, kind, g_t, b_t)[1], g_t)

    def L(kind=kind):
        return float((R * norm_forward(a_t, kind, g_t, b_t)[0]).sum())

    errs = [rel_err(da, numgrad(L, a_t)), rel_err(dg, numgrad(L, g_t))]
    errs += [] if kind == "rms" else [rel_err(dbe, numgrad(L, b_t))]  # RMSNorm không có β
    worst = max(worst, *errs)
    print(f"{LABEL[kind]:>9}: sai lệch tương đối của dL/da, dL/dγ" + ("" if kind == "rms" else ", dL/dβ"), end=": ")
    print(" · ".join(sci(e) for e in errs))
assert worst < 1e-7, "chiều ngược sai: hãy kiểm tra lại công thức"

Sai lệch tương đối cỡ $10^{-10}$ (giới hạn của sai phân hữu hạn), cho cả ba cách và cả ba gradient: công thức đúng.

Để ý dòng `dbeta = dy.sum(0)` và số hạng $\overline g$: với BatchNorm, $\partial L/\partial a$ cộng theo cột luôn
bằng 0. Vậy độ lệch $b$ của lớp Linear ngay trước BatchNorm **không bao giờ nhận gradient**: BatchNorm trừ trung bình
nên xóa mọi $b$, và $\beta$ làm thay việc của nó (Ioffe & Szegedy, 2015, mục 3.2). Vì thế người ta thường bỏ $b$
trước BatchNorm (`bias=False` trong PyTorch). Ta sẽ thấy điều này trên mạng thật ở mục 7.

## 6 · Chèn vào mạng, và đo sự “trôi dạt”

Mỗi lớp ẩn giờ là $a = hW + b$ → chuẩn hóa → ReLU, đúng như `reel/mnist_mlp/norm.py`, script tạo dữ liệu cho demo:
- SGD, batch 64, 1 lượt (epoch) qua 55.000 ảnh huấn luyện, tức 860 bước;
- seed $s$ quyết định cả trọng số đầu (`mulberry32(s)`) lẫn thứ tự ảnh (`default_rng(s)`);
- BatchNorm lúc huấn luyện dùng thống kê của lô, và cập nhật thêm một trung bình trượt mà mục 8 sẽ giải thích;
- một lần chạy “nổ” khi mất mát của một batch không còn là số hữu hạn.

Hàm `forward` có thêm tùy chọn `shake` cho thí nghiệm ở cuối mục này; tạm thời bỏ qua nó.

In [ ]:
def normalize(P, l, a, mode):
    """Lớp chuẩn hóa thứ l. mode = 'train': BatchNorm dùng thống kê của lô và cập nhật trung bình trượt; 'eval': dùng
    trung bình trượt; 'batch': thống kê của lô, không cập nhật. LayerNorm, RMSNorm làm như nhau ở mọi mode."""
    kind = P["norm"]
    if kind == "bn" and mode == "eval":
        xhat = (a - P["rm"][l]) * (1 / np.sqrt(P["rv"][l] + EPS))
        return P["g"][l] * xhat + P["be"][l], None
    if kind == "bn" and mode == "train":
        m = len(a)
        P["rm"][l] = (1 - MOM) * P["rm"][l] + MOM * a.mean(0)
        P["rv"][l] = (1 - MOM) * P["rv"][l] + MOM * a.var(0) * m / (m - 1)  # phương sai không chệch, như PyTorch
    return norm_forward(a, kind, P["g"][l], P["be"][l])


def forward(P, x, mode="train", shake=None):
    h, cache = x, []
    for l in range(P["depth"]):
        a = h @ P["W"][l] + P["b"][l]
        y, c = (a, None) if P["norm"] == "none" else normalize(P, l, a, mode)
        s = 1.0
        if shake:  # (mức nhiễu, rng): nhân và cộng mỗi nơ-ron với một số ngẫu nhiên mới ở mỗi bước
            level, rng = shake
            s = np.exp(level * rng.normal(size=WIDTH))
            y = y * s + level * rng.normal(size=WIDTH)
        cache.append((h, c, y, s))
        h = np.maximum(0, y)
    return cache, h, h @ P["W"][-1] + P["b"][-1]


def softmax(z):
    e = np.exp(z - z.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def cross_entropy(z, y):
    zmax = z.max(1)
    return float((zmax + np.log(np.exp(z - zmax[:, None]).sum(1)) - z[np.arange(len(y)), y]).mean())


def backward(P, cache, h, z, y):
    """Gradient theo W, b, γ, β của mọi lớp (Bài 06, cộng thêm norm_backward)."""
    dz = softmax(z)
    dz[np.arange(len(y)), y] -= 1
    dz /= len(y)
    G = {"W": [h.T @ dz], "b": [dz.sum(0)], "g": [], "be": []}
    dh = dz @ P["W"][-1].T
    for l in range(P["depth"] - 1, -1, -1):
        h_in, c, y_l, s = cache[l]
        dy = dh * (y_l > 0) * s  # qua ReLU (và qua nhiễu nhân, nếu có)
        if P["norm"] == "none":
            da = dy
        else:
            da, dg, dbe = norm_backward(dy, P["norm"], c, P["g"][l])
            G["g"].insert(0, dg)
            G["be"].insert(0, dbe)
        G["W"].insert(0, h_in.T @ da)
        G["b"].insert(0, da.sum(0))
        dh = da @ P["W"][l].T
    return G


def train(norm, depth, eta, seed=1, steps=860, shake=0.0, watch=None):
    """SGD với batch 64 trên ảnh huấn luyện theo thứ tự của seed. Trả về (mạng, bước nổ hoặc None).
    watch(step, P, G), nếu có, được gọi ở bước 0 và sau mỗi bước cập nhật."""
    P = init(norm, depth, seed)
    order = np.random.default_rng(seed).permutation(N_TRAIN)
    rng = np.random.default_rng(1000 + seed) if shake else None
    if watch:
        watch(0, P, None)
    with np.errstate(all="ignore"):  # một lần chạy nổ sẽ tràn số; ta tự phát hiện bên dưới
        for step in range(1, steps + 1):
            idx = order[(step - 1) * BATCH : step * BATCH]
            cache, h, z = forward(P, Xtr[idx], "train", (shake, rng) if shake else None)
            if not np.isfinite(cross_entropy(z, ytr[idx])):
                return P, step
            G = backward(P, cache, h, z, ytr[idx])
            for k in ("W", "b", "g", "be"):
                if k == "be" and norm == "rms":
                    continue  # RMSNorm không có β
                for p, g in zip(P[k], G[k]):
                    p -= eta * g
            if watch:
                watch(step, P, G)
    return P, None


def accuracy(P, X, y, mode="eval"):
    hits = [forward(P, X[i : i + 2000], mode)[2].argmax(1) == y[i : i + 2000] for i in range(0, len(X), 2000)]
    return float(np.concatenate(hits).mean())


def dead_share(P):
    """Tỉ lệ nơ-ron ẩn cho ra 0 với cả 500 ảnh kiểm định đầu (ReLU “chết hẳn”), như demo."""
    cache = forward(P, Xval[:500], "eval")[0]
    return float(np.mean([(y <= 0).all(0).mean() for _, _, y, _ in cache]))


print(f"1 epoch = {math.ceil(N_TRAIN / BATCH)} bước; bước cuối chỉ có {N_TRAIN % BATCH} ảnh, như trong script của demo")

Giờ ta đo điều mà bài báo BatchNorm gọi là **dịch chuyển hiệp biến nội tại** (internal covariate shift): phân bố đầu
vào của một lớp thay đổi khi các lớp trước nó được cập nhật. Ta huấn luyện hai mạng 10 lớp, seed 1, mỗi mạng với
một $\eta$ mà nó học tốt (theo bảng quét của demo):
- không chuẩn hóa, $\eta = 0{,}03$;
- BatchNorm, $\eta = 0{,}3$. Đây cũng là mạng mà demo dùng cho thẻ “Chuẩn hóa theo trục nào?”, nên mục 7 dùng lại nó.

Cứ mỗi 43 bước, ta đưa 500 ảnh kiểm định qua mạng và ghi trung bình, độ lệch chuẩn của $a$ ở lớp 5 cho 4 nơ-ron
đầu. Với mạng BatchNorm, ta ghi thêm đầu ra $y$ của lớp chuẩn hóa, tức thứ mà lớp sau thật sự nhận (chuẩn hóa bằng
thống kê của chính 500 ảnh đó).

In [ ]:
def drift_probe(log):
    def watch(step, P, G):
        if step % 43:
            return
        cache = forward(P, Xval[:500], "batch")[0]  # "batch": không đụng tới trung bình trượt
        h_in, _, y, _ = cache[4]
        a = h_in @ P["W"][4] + P["b"][4]
        log.append((step, a.mean(0)[:4], a.std(0)[:4], y.mean(0)[:4], y.std(0)[:4]))

    return watch


drift = {"none": [], "bn": []}
P_none, _ = train("none", 10, 0.03, watch=drift_probe(drift["none"]))
P_bn, _ = train("bn", 10, 0.3, watch=drift_probe(drift["bn"]))
for kind in drift:
    (s0, m0, sd0, ym0, ysd0), (s1, m1, sd1, ym1, ysd1) = drift[kind][0], drift[kind][-1]
    print(f"{LABEL[kind]}: a của 4 nơ-ron, bước 0 → bước {s1}")
    print("   trung bình:", " · ".join(f"{vn(u)} → {vn(v)}" for u, v in zip(m0, m1)))
    print("   độ lệch chuẩn:", " · ".join(f"{vn(u)} → {vn(v)}" for u, v in zip(sd0, sd1)))
    if kind == "bn":
        print("   đầu ra y sau BatchNorm, trung bình:", " · ".join(f"{vn(u)} → {vn(v)}" for u, v in zip(ym0, ym1)))
        print("   đầu ra y sau BatchNorm, độ lệch chuẩn:", " · ".join(f"{vn(u)} → {vn(v)}" for u, v in zip(ysd0, ysd1)))
val_none, val_bn = accuracy(P_none, Xval, yval), accuracy(P_bn, Xval, yval)
print(f"sau 1 epoch, đúng kiểm định: không chuẩn hóa {pct(val_none)} · BatchNorm {pct(val_bn)}")

Hình dưới vẽ các con số đó theo từng bước. Hàng trên là trung bình, hàng dưới là độ lệch chuẩn trên 500 ảnh kiểm
định. Mỗi nơ-ron một màu và một dạng điểm. Ba cột là $a$ của mạng không chuẩn hóa, $a$ của mạng BatchNorm (trước
chuẩn hóa), và $y$ của mạng BatchNorm (sau chuẩn hóa, thứ lớp sau nhận).

In [ ]:
NEURON = [(BLUE, "o"), (ORANGE, "s"), (GREEN, "^"), (GREY, "D")]
panels = [
    ("Không chuẩn hóa: a", drift["none"], 1, 2),
    ("BatchNorm: a (trước chuẩn hóa)", drift["bn"], 1, 2),
    ("BatchNorm: y (sau chuẩn hóa)", drift["bn"], 3, 4),
]
fig, axes = plt.subplots(2, 3, figsize=(10, 4.6), sharex=True, sharey="row")
for j, (title, log, im, isd) in enumerate(panels):
    steps = [r[0] for r in log]
    for k, (color, mk) in enumerate(NEURON):
        for row, idx in ((0, im), (1, isd)):
            axes[row, j].plot(
                steps, [r[idx][k] for r in log], color=color, marker=mk, ms=3.5, lw=1.3, label=f"nơ-ron {k + 1}"
            )
    axes[0, j].set_title(title, fontsize=10)
    axes[1, j].set_xlabel("bước SGD")
    for row in (0, 1):
        axes[row, j].grid(alpha=0.3)
axes[0, 0].set_ylabel("trung bình")
axes[1, 0].set_ylabel("độ lệch chuẩn")
axes[1, 0].legend(frameon=False, fontsize=8, ncol=2)
vn_axes(fig)
plt.tight_layout()
plt.show()

Ở mạng không chuẩn hóa, độ lệch chuẩn của cả bốn nơ-ron tăng nhiều lần trong một epoch, và trung bình cũng dời đi.
Lớp 6 phải học trên một đầu vào cứ đổi thang đo liên tục. Ở mạng BatchNorm, $a$ vẫn trôi (cột giữa), nhưng $y$ thì
được giữ ở trung bình gần 0 và độ lệch chuẩn gần 1 (cột phải): BatchNorm chặn được hai mô-men đầu của sự trôi.

Ioffe & Szegedy (2015) cho rằng đó chính là lý do BatchNorm giúp huấn luyện. Santurkar et al. (2018) kiểm tra giả
thuyết này bằng một thí nghiệm ngược: cố tình **thêm** trôi dạt ngay sau BatchNorm. Ta làm lại thí nghiệm đó. Ở mỗi
bước, mỗi nơ-ron được nhân với một số $e^{0{,}3\,z_1}$ và cộng thêm $0{,}3\,z_2$, với $z_1, z_2 \sim \mathcal N(0, 1)$
rút mới ở mỗi bước. Khi đánh giá thì không có nhiễu.

In [ ]:
z = np.random.default_rng(0).normal(size=(2, 100_000))
lo, hi = np.quantile(np.exp(0.3 * z[0]), [0.025, 0.975])
print(f"nhiễu mức 0,3: 95% số lần, hệ số nhân nằm trong [{vn(lo)}; {vn(hi)}], số cộng trong ±{vn(0.3 * 1.96)}")
y_shift = max(abs(v) for r in drift["bn"] for v in r[3])
y_sd = [v for r in drift["bn"] for v in r[4]]
print(f"trôi tự nhiên của y sau BatchNorm trong cả epoch: |trung bình| ≤ {vn(y_shift)}, ", end="")
print(f"độ lệch chuẩn {vn(min(y_sd))}–{vn(max(y_sd))}")

### 🤔 Dự đoán trước
Nhiễu này tạo ra mức trôi lớn hơn nhiều lần mức trôi tự nhiên vừa đo, và đổi ở mọi bước. Nếu BatchNorm giúp vì nó
chặn trôi dạt, thì mạng BatchNorm có nhiễu sẽ học kém đi nhiều. Bạn đoán độ chính xác kiểm định sẽ giảm từ mức của
BatchNorm sạch xuống gần mức nào: gần mạng không chuẩn hóa, gần đoán mò, hay gần như không đổi?

In [ ]:
P_shake, boom = train("bn", 10, 0.3, shake=0.3)
val_shake = accuracy(P_shake, Xval, yval)
print(f"BatchNorm, η = 0,3, có nhiễu mức 0,3 sau mỗi lớp chuẩn hóa: đúng kiểm định {pct(val_shake)}")
print(f"BatchNorm sạch: {pct(val_bn)} · không chuẩn hóa (η = 0,03): {pct(val_none)}")

Gần như không đổi: thêm nhiễu mạnh ngay sau mỗi lớp chuẩn hóa chỉ làm mạng BatchNorm kém đi chút ít, và nó vẫn hơn
mạng không chuẩn hóa vừa huấn luyện (một seed; mục 12 thử thêm hai seed). Mạng này học tốt dù phân bố đầu vào của mọi
lớp bị xáo trộn ở **từng bước**, nên việc chặn trôi dạt không thể là toàn bộ lý do BatchNorm giúp. Santurkar et al. (2018) thấy đúng như vậy trên mạng ảnh lớn (VGG trên
CIFAR-10). Họ đề xuất một lời giải thích khác: BatchNorm làm **địa hình mất mát trơn hơn**, nên gradient ổn định hơn
và chịu được bước lớn hơn. Đến nay vì sao BatchNorm hiệu quả vẫn còn tranh luận; mục 9 đo một cơ chế cụ thể.

## 7 · Chuẩn hóa theo trục nào? Trên kích hoạt thật

Mạng BatchNorm vừa huấn luyện (10 lớp, $\eta = 0{,}3$, seed 1) chính là mạng của thẻ đầu tiên trong demo. Ô dưới lấy
đầu vào $a$ của lớp chuẩn hóa thứ 5 cho 8 ảnh kiểm tra đầu tiên, rồi so với số mà demo lưu (8 ảnh × 8 nơ-ron đầu).

In [ ]:
pan = sweep["panel"]
h_in = forward(P_bn, Xte[:8], "eval")[0][4][0]  # đầu vào của lớp ẩn 5 (sau ReLU của lớp 4)
A = h_in @ P_bn["W"][4] + P_bn["b"][4]  # 8 ảnh × 64 nơ-ron, trước chuẩn hóa
test_bn = accuracy(P_bn, Xte, yte)
print(
    f"mạng BatchNorm: đúng kiểm định {pct(val_bn)}, kiểm tra {pct(test_bn)} (demo: {pct(pan['val'])}, {pct(pan['testRunning'])})"
)
print("a của lớp 5, 8 ảnh × 8 nơ-ron đầu; sai khác lớn nhất so với demo:", vn(np.abs(A[:, :8] - pan["a"]).max(), 5))
assert abs(val_bn - pan["val"]) < 5e-4 and abs(test_bn - pan["testRunning"]) < 5e-4, "khác với demo"
assert np.allclose(A[:, :8], pan["a"], atol=1e-3), "khác với demo: hãy kiểm tra lại dữ liệu"
for key, arr in [
    ("runMean", P_bn["rm"][4]),
    ("runVar", P_bn["rv"][4]),
    ("gamma", P_bn["g"][4]),
    ("beta", P_bn["be"][4]),
]:
    assert np.allclose(arr[:8], pan[key], atol=1e-3), f"{key} khác với demo"
print("trung bình trượt, phương sai trượt, γ, β của lớp 5 cũng trùng với demo")

Hình dưới tô màu cùng 8 × 8 ô đó trước và sau ba phép chuẩn hóa: xanh là dương, cam là âm, càng đậm càng xa 0, và
mỗi ô ghi giá trị của nó. Khung đen bao các ô dùng chung một cặp $\mu, \sigma$ với ô góc trên trái:
- BatchNorm chuẩn hóa theo **cột**, trên lô 8 ảnh này;
- LayerNorm và RMSNorm chuẩn hóa theo **hàng**, trên cả 64 nơ-ron của mỗi ảnh (hình chỉ vẽ 8 nơ-ron đầu; demo tính
  trên 8 nơ-ron hiển thị).

In [ ]:
views = [
    ("a (chưa chuẩn hóa)", A, None),
    ("BatchNorm, lô 8 ảnh: theo cột", norm_forward(A, "bn", one, zero)[0], "col"),
    ("LayerNorm: theo hàng", norm_forward(A, "ln", one, zero)[0], "row"),
    ("RMSNorm: theo hàng, không trừ μ", norm_forward(A, "rms", one, zero)[0], "row"),
]
vmax = max(np.abs(v[:, :8]).max() for _, v, _ in views)
fig, axes = plt.subplots(2, 2, figsize=(9.6, 8.4))
for ax, (title, v, group) in zip(axes.flat, views):
    im = ax.imshow(v[:, :8], cmap=DIVERGING, vmin=-vmax, vmax=vmax)
    for i in range(8):
        for j in range(8):
            ax.text(j, i, vn(v[i, j], 2), ha="center", va="center", fontsize=7)
    if group:  # khung: các ô dùng chung μ, σ với ô (1, 1)
        w, hgt = (1, 8) if group == "col" else (8, 1)
        ax.add_patch(plt.Rectangle((-0.5, -0.5), w, hgt, fill=False, ec="black", lw=2.2))
    ax.set_xticks(range(8), [f"n{j + 1}" for j in range(8)], fontsize=8)
    ax.set_yticks(range(8), [f"ảnh {i + 1} ({yte[i]})" for i in range(8)], fontsize=8)
    ax.set_title(title, fontsize=10)
fig.colorbar(im, ax=axes, fraction=0.03, pad=0.02, label="giá trị")
vn_axes(fig)
plt.show()
for title, v, _ in views:
    neg = [f"n{j + 1}" for j in range(8) if (v[:, j] < 0).all()]
    pos = [f"n{j + 1}" for j in range(8) if (v[:, j] > 0).all()]
    print(
        f"{title}: cột âm với cả 8 ảnh: {', '.join(neg) or 'không có'}; cột dương với cả 8 ảnh: {', '.join(pos) or 'không có'}"
    )

Ở ô trên trái, mỗi cột có thang đo riêng: cột n5 âm (cam) với cả 8 ảnh, còn cột n3 có vài ô lớn gấp nhiều lần các ô
khác. Sau BatchNorm, trong lô 8 ảnh này, không còn cột nào cùng một dấu cho cả 8 ảnh: mỗi nơ-ron được đặt về trung
bình 0 *so với các ảnh khác trong lô*. Sau LayerNorm hay RMSNorm, mỗi hàng được đặt về cùng một thang đo, nhưng cột n5
vẫn âm với cả 8 ảnh. (Tám ô
của một hàng không có trung bình 0 vì phép tính dùng cả 64 nơ-ron, hình chỉ vẽ 8.)

Hệ quả của trục: kết quả BatchNorm của một ảnh **phụ thuộc vào các ảnh khác trong lô**, còn LayerNorm và RMSNorm
thì không. Vì vậy LayerNorm hợp với Transformer, nơi mỗi token được chuẩn hóa riêng và lô có thể rất nhỏ.

### γ, β: trả lại tự do cho mạng

Ép mọi nơ-ron về trung bình 0, phương sai 1 có thể làm mất thứ mà mạng cần. Vì vậy mỗi nơ-ron có thêm $\gamma$ và
$\beta$ học được. Ioffe & Szegedy (2015) chỉ ra rằng với BatchNorm, đặt $\gamma = \sqrt{\sigma^2 + \epsilon}$ và
$\beta = \mu$ sẽ trả lại đúng giá trị cũ. Ô dưới kiểm tra điều đó với $\mu, \sigma^2$ là trung bình trượt (mục 8),
xem mạng đã học $\gamma, \beta$ thế nào, và thử làm điều tương tự với LayerNorm.

In [ ]:
xhat_bn = (A - P_bn["rm"][4]) / np.sqrt(P_bn["rv"][4] + EPS)
undo = np.sqrt(P_bn["rv"][4] + EPS) * xhat_bn + P_bn["rm"][4]
print(f"BatchNorm với γ = √(σ² + ε), β = μ: sai khác lớn nhất so với a = {sci(np.abs(undo - A).max())}")
assert np.abs(undo - A).max() < 1e-9
gs, bs = np.concatenate(P_bn["g"]), np.concatenate(P_bn["be"])
print(f"γ đã học (10 lớp × 64 nơ-ron): {vn(gs.min())} đến {vn(gs.max())}; β: {vn(bs.min())} đến {vn(bs.max())}")

# LayerNorm: γ, β thuộc về từng nơ-ron, còn μ, σ thuộc về từng ảnh. Tìm γ, β tốt nhất (bình phương tối thiểu) trên
# 10.000 ảnh kiểm tra để đưa x̂ của LayerNorm về lại a:
A_all = forward(P_bn, Xte, "eval")[0][4][0] @ P_bn["W"][4] + P_bn["b"][4]
xl = norm_forward(A_all, "ln", one, zero)[0]
g_fit = ((xl - xl.mean(0)) * (A_all - A_all.mean(0))).mean(0) / xl.var(0)
b_fit = A_all.mean(0) - g_fit * xl.mean(0)
resid = np.linalg.norm(A_all - (g_fit * xl + b_fit)) / np.linalg.norm(A_all - A_all.mean(0))
print(f"LayerNorm với γ, β tốt nhất cho từng nơ-ron: vẫn sai {pct(resid, 0)} (so với độ dao động của a)")

Với BatchNorm, việc “hoàn tác” là chính xác, nên chuẩn hóa không lấy mất gì của mạng: nếu cần, nó có thể học lại
giá trị cũ. Mạng thật thì không làm vậy: sau 1 epoch, $\gamma$ vẫn ở quanh 1 và $\beta$ quanh 0, tức là nó *giữ*
chuẩn hóa. Với LayerNorm thì không hoàn tác được: mỗi ảnh mất đi trung bình và độ lệch chuẩn riêng của nó, còn
$\gamma, \beta$ chỉ có một giá trị cho mỗi nơ-ron, dùng chung cho mọi ảnh.

Một chi tiết nữa từ mục 5: độ lệch $b$ của lớp Linear ngay trước BatchNorm không bao giờ nhận gradient. Kiểm tra trên
mạng vừa huấn luyện 860 bước:

In [ ]:
b_max = max(np.abs(b).max() for b in P_bn["b"][:-1])  # 10 lớp ẩn; lớp ra không có BatchNorm
print(
    f"|b| lớn nhất ở 10 lớp ẩn sau 860 bước: {sci(b_max)} (chỉ là sai số làm tròn); lớp ra: {vn(np.abs(P_bn['b'][-1]).max(), 3)}"
)
assert b_max < 1e-12

## 8 · BatchNorm khi huấn luyện và khi suy luận

Lúc huấn luyện, BatchNorm dùng $\mu, \sigma^2$ của lô 64 ảnh. Lúc suy luận, ta muốn dự đoán cho một ảnh chỉ phụ thuộc
vào chính ảnh đó. Vì vậy, trong lúc huấn luyện, mỗi lớp còn giữ một **trung bình trượt**:

$$\mu_{\text{trượt}} \leftarrow 0{,}9\,\mu_{\text{trượt}} + 0{,}1\,\mu_{\text{lô}} \qquad
\sigma^2_{\text{trượt}} \leftarrow 0{,}9\,\sigma^2_{\text{trượt}} + 0{,}1\,\tfrac{m}{m-1}\sigma^2_{\text{lô}}$$

với $m = 64$ ảnh mỗi lô; hệ số $\tfrac{m}{m-1}$ cho phương sai không chệch, như Ioffe & Szegedy (2015) và PyTorch.
Hệ số 0,1 được PyTorch gọi là `momentum`, dù nó là trọng số của giá trị *mới*, ngược với momentum của Bài 07. Khi
suy luận (`model.eval()` trong PyTorch), BatchNorm dùng hai giá trị trượt đó: chế độ `"eval"` của hàm `normalize`.

Ô dưới viết lại cách suy luận *sai*: vẫn dùng thống kê của lô, với các lô gồm $B$ ảnh kiểm tra liên tiếp.

In [ ]:
def predict_batch_stats(P, X, B):
    """Logit của mạng BatchNorm khi suy luận sai cách: mỗi nhóm B ảnh liên tiếp được chuẩn hóa bằng thống kê của chính nó."""
    h, n = X, len(X) // B * B
    for l in range(P["depth"]):
        a = h @ P["W"][l] + P["b"][l]
        groups = [a[:n].reshape(-1, B, WIDTH)] + ([a[n:][None]] if n < len(a) else [])  # nhóm cuối có thể ít hơn B
        xhat = [(g - g.mean(1, keepdims=True)) / np.sqrt(g.var(1, keepdims=True) + EPS) for g in groups]
        h = np.maximum(0, P["g"][l] * np.concatenate([x.reshape(-1, WIDTH) for x in xhat]) + P["be"][l])
    return h @ P["W"][-1] + P["b"][-1]


z64 = predict_batch_stats(P_bn, Xte, 64)
print(f"lô 64 ảnh, thống kê của lô: đúng {pct((z64.argmax(1) == yte).mean())} · trung bình trượt: đúng {pct(test_bn)}")

### 🤔 Dự đoán trước
Giờ ta quên luôn chế độ suy luận và đưa **từng ảnh một** (lô 1 ảnh) qua mạng, vẫn dùng thống kê của lô. Trên
10.000 ảnh kiểm tra, mạng đúng bao nhiêu phần trăm? (Gợi ý: với một ảnh, $\mu$ của mỗi nơ-ron là gì?)

In [ ]:
z1 = predict_batch_stats(P_bn, Xte, 1)
same = np.allclose(z1, z1[0])
print(f"lô 1 ảnh: mọi ảnh cho cùng một logit: {same}; mạng luôn đoán số {z1[0].argmax()}")
acc_b = {B: float((predict_batch_stats(P_bn, Xte, B).argmax(1) == yte).mean()) for B in (1, 2, 8, 64)}
for B, acc in acc_b.items():
    print(f"  thống kê của lô {B:>2} ảnh: đúng {pct(acc)} (demo: {pct(pan['testBatch'][str(B)])})")
print(f"  trung bình trượt (đúng cách): {pct(test_bn)}")
print(f"số ảnh kiểm tra là số {z1[0].argmax()}: {vn((yte == z1[0].argmax()).sum(), 0)} / {vn(len(yte), 0)}")
for B, acc in acc_b.items():
    assert abs(acc - pan["testBatch"][str(B)]) < 5e-4, "khác với demo: hãy kiểm tra lại dữ liệu"

Với một ảnh, $\mu$ chính là giá trị đó, nên $x - \mu = 0$ ở mọi nơ-ron: ảnh biến mất ngay ở lớp đầu tiên. Mọi ảnh
cho cùng một đầu ra (chỉ còn $\beta$ đi tiếp), nên mạng luôn đoán cùng một chữ số. Nó đúng đúng bằng tỉ lệ ảnh có
chữ số đó trong tập kiểm tra. Lô 2 ảnh cũng tệ: mỗi giá trị chuẩn hóa chỉ còn $\pm 1$, tức “lớn hơn hay nhỏ hơn ảnh
kia”. Lô càng nhỏ, thống kê càng nhiễu; ngay cả lô 64 ảnh vẫn kém trung bình trượt. Đây là lỗi thật hay gặp: quên
`model.eval()` trong PyTorch khiến mạng BatchNorm suy luận bằng thống kê của lô.

Trung bình trượt cũng chỉ là một ước lượng: nó tích lũy $\mu$ của những lô đã đi qua, khi trọng số còn đang thay đổi.
Wu & Johnson (2021) đề xuất “PreciseBN”: giữ nguyên trọng số rồi tính $\mu, \sigma^2$ thật trên nhiều ảnh huấn luyện.
Ô dưới làm điều đó trên cả 55.000 ảnh, từng lớp một:

In [ ]:
P_precise = {**P_bn, "rm": [], "rv": []}
h = Xtr
for l in range(10):
    a = h @ P_bn["W"][l] + P_bn["b"][l]
    P_precise["rm"].append(a.mean(0))
    P_precise["rv"].append(a.var(0))
    h = np.maximum(0, P_bn["g"][l] * (a - a.mean(0)) / np.sqrt(a.var(0) + EPS) + P_bn["be"][l])
test_precise = accuracy(P_precise, Xte, yte)
d_mu = np.abs(P_precise["rm"][4] - P_bn["rm"][4]).max()
print(f"lớp 5: trung bình trượt lệch tối đa {vn(d_mu, 3)} so với trung bình thật trên tập huấn luyện")
print(f"đúng kiểm tra: trung bình trượt {pct(test_bn)} · thống kê thật trên 55.000 ảnh {pct(test_precise)}", end=" ")
print(f"(chênh {vn(100 * (test_precise - test_bn), 2)} điểm phần trăm)")

Thống kê thật cho kết quả gần như bằng trung bình trượt (chênh 0,16 điểm phần trăm trên một mạng, một seed): ở
đây trung bình trượt đã là một ước lượng đủ tốt. LayerNorm và RMSNorm không có vấn đề này: chúng làm **đúng cùng một
phép tính** khi huấn luyện và khi suy luận (Ba et al., 2016).

## 9 · Trên mạng sâu thật: chuẩn hóa giúp tới đâu?

Demo đã huấn luyện cả bốn cách × độ sâu 10/20/30 × 6 giá trị $\eta$ × 3 seed, mỗi lần 1 epoch. Trước khi đọc kết quả
đó, ta chạy lại một hàng của nó: mạng 10 lớp, seed 1, $\eta = 1$, một tốc độ học lớn. Ở mục 6, mạng không chuẩn hóa
học tốt với $\eta = 0{,}03$ và mạng BatchNorm với $\eta = 0{,}3$. Với mỗi lần chạy ta in độ chính xác kiểm định và tỉ
lệ nơ-ron ReLU “chết hẳn” (cho ra 0 với cả 500 ảnh kiểm định đầu).

In [ ]:
def stored(norm, depth, eta, seed=1):
    run = next(r for r in sweep["runs"] if (r["norm"], r["depth"], r["eta"]) == (norm, depth, eta))
    return run["seeds"][sweep["seeds"].index(seed)]


def show(r):
    return f"nổ ở bước {r['nanStep']}" if "nanStep" in r else f"đúng {pct(r['val'])}, nơ-ron chết {pct(r['dead'], 1)}"


live, nets = {}, {}
for norm in NORMS:
    P, boom = train(norm, 10, 1.0)
    nets[norm] = P
    live[norm] = {"nanStep": boom} if boom else {"val": accuracy(P, Xval, yval), "dead": dead_share(P)}
    print(f"{LABEL[norm]:>15}, η = 1: {show(live[norm]):<36} demo: {show(stored(norm, 10, 1))}")
for norm in ("ln", "rms"):
    guess = np.bincount(forward(nets[norm], Xval, "eval")[2].argmax(1), minlength=10)
    print(f"{LABEL[norm]}, η = 1 đoán: {', '.join(f'số {d}: {vn(n, 0)} ảnh' for d, n in enumerate(guess) if n)}")
print(f"số ảnh kiểm định là số 1: {(yval == 1).sum()} / {vn(len(yval), 0)}")
fail = "nanStep" in live["none"] or live["none"]["val"] < 0.15  # nổ, hoặc kẹt ở mức đoán mò
assert fail and abs(live["bn"]["val"] - stored("bn", 10, 1)["val"]) < 0.005, "khác với demo"
assert abs(val_none - stored("none", 10, 0.03)["val"]) < 5e-4, "khác với demo"  # hai mạng của mục 6
for norm in ("ln", "rms"):
    assert abs(live[norm]["val"] - stored(norm, 10, 1)["val"]) < 5e-4, "khác với demo"
    assert abs(live[norm]["dead"] - stored(norm, 10, 1)["dead"]) < 0.01, "khác với demo"

Ba cách phản ứng rất khác nhau với cùng một $\eta = 1$:
- **Không chuẩn hóa:** nổ (ở bước 182 với seed 1; trên một máy khác, sai số làm tròn có thể khiến nó không nổ mà
  kẹt ở mức đoán mò, như phép kiểm tra `fail` ở trên cho phép).
- **BatchNorm:** vẫn học tốt, và không có nơ-ron nào chết.
- **LayerNorm và RMSNorm:** không nổ, nhưng sụp: khoảng một nửa số nơ-ron ReLU chết hẳn, và mạng đoán “1” cho mọi ảnh.
  Độ chính xác của nó đúng bằng tỉ lệ số 1 trong tập kiểm định.

Vì sao nơ-ron chết ở LayerNorm mà không chết ở BatchNorm? BatchNorm ép **mỗi nơ-ron** có trung bình 0 trên lô. Khi
$\gamma, \beta$ còn gần 1 và 0, luôn có ảnh làm nó dương, nên nó không thể chết hẳn. LayerNorm chỉ ép **mỗi ảnh** có
trung bình 0 trên các nơ-ron của nó. Một nơ-ron vẫn có thể âm với mọi ảnh, và khi đó ReLU chặn luôn gradient của nó.

Vì sao BatchNorm không nổ? Ioffe & Szegedy (2015, mục 3.3) chỉ ra rằng BatchNorm **không đổi khi nhân $W$ với một
số**: $\mathrm{BN}(h \cdot cW) = \mathrm{BN}(hW)$ với mọi $c > 0$ (bỏ qua $\epsilon$ và $b$, mà $b$ ở đây luôn bằng 0).
Còn gradient theo $W$ thì nhỏ đi $c$ lần. Ô dưới kiểm tra cả hai điều trên mạng BatchNorm của mục 7, rồi xem các bước
lớn đã làm gì với độ lớn của $W$ trong lần chạy $\eta = 1$:

In [ ]:
xb, yb = Xtr[:64], ytr[:64]
P3 = {**P_bn, "W": [w * (3 if l == 4 else 1) for l, w in enumerate(P_bn["W"])]}  # nhân W₅ với 3
grads = []
for P in (P_bn, P3):
    cache, h, z = forward(P, xb, "batch")
    grads.append((z, backward(P, cache, h, z, yb)["W"][4]))
(z_a, g_a), (z_b, g_b) = grads
norm_ratio = np.linalg.norm(g_a) / np.linalg.norm(g_b)
print(f"nhân W₅ với 3: logit đổi tối đa {sci(np.abs(z_a - z_b).max())}; ‖∂L/∂W₅‖ nhỏ đi {vn(norm_ratio, 4)} lần")
assert np.abs(z_a - z_b).max() < 1e-4 and abs(norm_ratio - 3) < 1e-3
P0 = init("bn", 10)
growth = [np.linalg.norm(w) / np.linalg.norm(w0) for w, w0 in zip(nets["bn"]["W"], P0["W"])]
print("BatchNorm, η = 1: ‖W‖ sau 1 epoch / lúc đầu, lớp 1…10:", " · ".join(vn(g, 2) for g in growth[:10]))

Logit gần như không đổi (chỉ khác ở mức sai số làm tròn và $\epsilon$), còn gradient nhỏ đúng 3 lần. Điều này giải
thích lần chạy $\eta = 1$. Một bước lớn làm $W$ dài ra (lớp 1 dài gấp đôi sau 1 epoch), nhưng mạng vẫn tính đúng như
cũ, còn gradient thì nhỏ lại theo. Bước tương đối $\|\Delta W\| / \|W\|$ vì thế nhỏ đi theo bình phương độ dài của $W$:
BatchNorm **tự giảm tốc độ học hiệu dụng** khi bước quá lớn (Arora et al., 2019). Mạng không chuẩn hóa không có cơ chế
này: $W$ dài ra thì kích hoạt và gradient cũng lớn theo, rồi nổ. Bjorck et al. (2018) đo được rằng lợi ích chính
của BatchNorm là cho phép dùng $\eta$ lớn hơn.

### 🤔 Dự đoán trước
Ở 10 lớp, BatchNorm đúng là giúp. Ở 30 lớp, với $\eta$ tốt nhất cho mỗi cách, BatchNorm hơn hay kém mạng không chuẩn
hóa? Còn LayerNorm?

In [ ]:
def summary(norm, depth, eta):
    """Trung vị 3 seed (lần nổ tính là tệ nhất, như demo) và khoảng min–max của các lần không nổ."""
    seeds = next(r for r in sweep["runs"] if (r["norm"], r["depth"], r["eta"]) == (norm, depth, eta))["seeds"]
    ok = [s["val"] for s in seeds if "val" in s]
    med = sorted(s.get("val", -1.0) for s in seeds)[1]
    return {
        "med": med if med >= 0 else None,
        "lo": min(ok, default=None),
        "hi": max(ok, default=None),
        "nan": 3 - len(ok),
    }


def best(norm, depth):
    """η có trung vị cao nhất (hòa thì lấy η nhỏ hơn, như demo)."""
    cands = [(e, summary(norm, depth, e)) for e in sweep["etas"]]
    return max((c for c in cands if c[1]["med"] is not None), key=lambda c: c[1]["med"])


for depth in sweep["depths"]:
    print(f"{depth} lớp ẩn, η tốt nhất của mỗi cách (trung vị 3 seed, khoảng min–max):")
    for norm in NORMS:
        eta, s = best(norm, depth)
        print(f"  {LABEL[norm]:>15}: η = {vn_eta(eta):<4} {pct(s['med'])} ({pct(s['lo'])}–{pct(s['hi'])})")
    n = best("none", depth)[1]
    for norm in ("bn", "ln", "rms"):
        b = best(norm, depth)[1]
        gap = "khoảng 3 seed tách rời" if b["lo"] > n["hi"] or n["lo"] > b["hi"] else "khoảng 3 seed chồng lên nhau"
        print(f"  {LABEL[norm]} − không chuẩn hóa: {vn(100 * (b['med'] - n['med']), 2)} điểm phần trăm, {gap}")
top_bn30 = max(s["val"] for r in sweep["runs"] if (r["norm"], r["depth"]) == ("bn", 30) for s in r["seeds"])
print(f"lần chạy tốt nhất của BatchNorm 30 lớp, mọi η và seed: {pct(top_bn30)}")

Hình dưới vẽ cả lưới, như thẻ thứ hai của demo: độ chính xác kiểm định theo $\eta$ (thang log), mỗi độ sâu một ô.
Điểm là trung vị 3 seed, vạch dọc là min–max của các lần không nổ, dấu × ở đáy là $\eta$ làm ít nhất 2/3 seed nổ.
Mỗi cách một màu, một dạng điểm và một kiểu nét; các cách được dời nhẹ sang hai bên để khỏi che nhau.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.6), sharey=True)
for ax, depth in zip(axes, sweep["depths"]):
    for k, norm in enumerate(NORMS):
        st, shift = STYLE[norm], 1.09 ** (k - 1.5)  # dời theo thang log
        rows = [(e * shift, summary(norm, depth, e)) for e in sweep["etas"]]
        ok = [(x, s) for x, s in rows if s["med"] is not None]
        ax.plot([x for x, _ in ok], [100 * s["med"] for _, s in ok], color=st["color"], ls=st["ls"], lw=1.5)
        for x, s in ok:
            ax.plot([x, x], [100 * s["lo"], 100 * s["hi"]], color=st["color"], lw=1.2)
        ax.plot(
            [x for x, _ in ok],
            [100 * s["med"] for _, s in ok],
            st["marker"],
            color=st["color"],
            mfc=st["mfc"],
            ms=6,
            label=LABEL[norm],
        )
        ax.plot(
            [x for x, s in rows if s["med"] is None],
            [3] * sum(s["med"] is None for _, s in rows),
            "x",
            color=st["color"],
            ms=8,
            mew=2,
        )
    ax.axhline(10, color=GREY, lw=0.8, ls=":")
    ax.set(xscale="log", title=f"{depth} lớp ẩn", xlabel="η (thang log)", ylim=(0, 100))
    ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("đúng kiểm định (%)")
axes[0].text(0.012, 12, "đoán mò 10%", fontsize=8, color=GREY)
axes[0].legend(frameon=False, fontsize=8, loc="lower left", bbox_to_anchor=(0, 0.12))
vn_axes(fig)
for ax in axes:
    ax.set_xticks(sweep["etas"], [vn_eta(e) for e in sweep["etas"]], fontsize=8)
    ax.minorticks_off()
plt.tight_layout()
plt.show()

Ở 10 và 20 lớp, BatchNorm thắng rõ: khoảng 3 seed của nó nằm hẳn trên khoảng của mạng không chuẩn hóa. LayerNorm cũng
hơn rõ nhưng ít hơn; RMSNorm hơn ít nhất (ở 10 lớp còn trong dao động giữa các seed). Ở 30 lớp thì không cách nào hơn
rõ mạng không chuẩn hóa. Trung vị của BatchNorm tụt từ 93,92% (20 lớp) xuống 65,96%, thấp hơn mạng không chuẩn hóa 11
điểm, và lần chạy tốt nhất của nó trên mọi $\eta$ và seed chỉ đạt 75,90%. Khoảng 3 seed của hai cách vẫn chồng lên
nhau, nên khoảng cách 11 điểm đó nằm **trong dao động giữa các seed**: ở $\eta$ tốt nhất của mỗi cách, ta chỉ nói được
là BatchNorm *không hơn*. Ô dưới so từng $\eta$ ở 30 lớp:

In [ ]:
print("30 lớp ẩn: BatchNorm so với không chuẩn hóa ở từng η (min–max 3 seed, lần nổ không tính)")
for eta in sweep["etas"]:
    b, n = summary("bn", 30, eta), summary("none", 30, eta)
    rng_b = f"{pct(b['lo'])}–{pct(b['hi'])}"
    rng_n = f"{pct(n['lo'])}–{pct(n['hi'])}" if n["lo"] is not None else "cả 3 seed nổ"
    if n["lo"] is None or b["lo"] > n["hi"]:
        verdict = "BatchNorm hơn rõ"
    elif b["hi"] < n["lo"]:
        verdict = "BatchNorm kém rõ"
    else:
        verdict = "trong dao động giữa các seed"
    print(f"  η = {vn_eta(eta):<4}: BatchNorm {rng_b:<16} không chuẩn hóa {rng_n:<16} → {verdict}")

Với $\eta$ lớn, BatchNorm vẫn giữ mạng không nổ và hơn hẳn. Nhưng với $\eta$ nhỏ, nơi mạng không chuẩn hóa học tốt
nhất, BatchNorm kém rõ ở $\eta = 0{,}01$. Vì sao một kỹ thuật giúp mạng 10 lớp lại không giúp mạng 30 lớp?

## 10 · Vì sao BatchNorm không cứu được mạng 30 lớp?

Yang et al. (2019) chứng minh rằng trong một mạng BatchNorm *thẳng* (không có kết nối tắt), ngay lúc khởi tạo,
gradient **tăng theo cấp số nhân** khi đi ngược về các lớp đầu. Đổi khởi tạo hay hàm kích hoạt đều không chữa được,
vì nguyên nhân là chính BatchNorm. Ta đo điều đó trên lô huấn luyện đầu tiên của seed 1, như demo: $\|\partial L/\partial W_l\|$
cho từng lớp, trước bước cập nhật nào.

In [ ]:
idx1 = np.random.default_rng(1).permutation(N_TRAIN)[:BATCH]  # lô đầu tiên của seed 1
x1, y1 = Xtr[idx1], ytr[idx1]


def layer_grads(norm, depth, x=x1, y=y1):
    """‖∂L/∂W_l‖ của từng lớp ẩn lúc khởi tạo (BatchNorm dùng thống kê của lô x)."""
    P = init(norm, depth)
    cache, h, z = forward(P, x, "train")
    return np.array([np.linalg.norm(g) for g in backward(P, cache, h, z, y)["W"][:depth]])


print("‖∂L/∂W₁‖ / ‖∂L/∂W_L‖ lúc khởi tạo (trong ngoặc: demo)")
for depth in sweep["depths"]:
    row = []
    for norm in NORMS:
        g = layer_grads(norm, depth)
        ref = sweep["initGrad"][norm][str(depth)]
        assert abs(g[0] / g[-1] - ref) < 0.006 * ref, "khác với demo: hãy kiểm tra lại dữ liệu"
        row.append(f"{LABEL[norm]} {vn(g[0] / g[-1], 1)} ({vn(ref, 1)})")
    print(f"  {depth} lớp: " + " · ".join(row))
prof = {norm: layer_grads(norm, 30) for norm in NORMS}

Hình dưới vẽ $\|\partial L/\partial W_l\|$ theo lớp $l$ của mạng 30 lớp, thang log. Đi từ phải sang trái là đi ngược
từ lớp ra về lớp đầu, đúng chiều lan truyền ngược.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
layers = np.arange(1, 31)
for norm in NORMS:
    st = STYLE[norm]
    ax.plot(
        layers,
        prof[norm],
        color=st["color"],
        ls=st["ls"],
        marker=st["marker"],
        mfc=st["mfc"],
        ms=4.5,
        lw=1.5,
        label=LABEL[norm],
    )
ax.set(yscale="log", xlabel="lớp ẩn l (1 = gần đầu vào)", ylabel="‖∂L/∂Wₗ‖ lúc khởi tạo", xlim=(0.5, 30.5))
ax.set_title("Mạng 30 lớp, lô đầu tiên của seed 1: gradient theo từng lớp", fontsize=11)
ax.grid(alpha=0.3, which="both")
ax.legend(frameon=False, fontsize=9, ncol=2)
vn_axes(fig)
plt.show()

Ở mạng không chuẩn hóa, LayerNorm và RMSNorm, gradient của các lớp có cùng cỡ: lớp đầu chỉ lớn hơn lớp cuối vài lần.
Ở mạng BatchNorm, đường xanh là một đường thẳng dốc trên thang log, tức là gradient **nhân lên một hệ số cố định mỗi
lớp** khi đi ngược. Ô dưới đo hệ số đó (độ dốc của đường thẳng khớp nhất qua các lớp 2–30) và so với dự đoán của
Yang et al. (2019, Định lý 3.10). Định lý tính cho mạng rộng vô hạn và cho bình phương chuẩn của gradient, nên hệ số của
chính chuẩn gradient là căn bậc hai của nó; khi lô rất lớn nó tiến tới $\sqrt{\pi/(\pi-1)}$.

In [ ]:
def yang_rate(B):
    """Hệ số tăng của ‖gradient‖² mỗi lớp trong mạng ReLU + BatchNorm, lô B (Yang et al., 2019, Định lý 3.10)."""
    c = -1 / (B - 1)
    J1 = (math.sqrt(1 - c * c) + (math.pi - math.acos(c)) * c) / math.pi  # nhân arccos (Cho & Saul, 2009)
    dJ1 = (math.pi - math.acos(c)) / math.pi
    return (B - 1 + dJ1) / ((B - 3) * (1 - J1))


k = np.arange(2, 31)
factor = {norm: float(np.exp(-np.polyfit(k, np.log(prof[norm][1:]), 1)[0])) for norm in NORMS}
print(
    "gradient nhân lên mỗi lớp khi đi ngược (lớp 30 → 2):", " · ".join(f"{LABEL[n]} {vn(factor[n], 3)}" for n in NORMS)
)
print(f"lý thuyết cho BatchNorm: lô 64 → {vn(math.sqrt(yang_rate(64)), 3)}; lô rất lớn → √(π/(π − 1)) = ", end="")
print(f"{vn(math.sqrt(math.pi / (math.pi - 1)), 3)}; mũ 29: {vn(factor['bn'] ** 29, 0)} lần")

Mạng thật (rộng 64, một lô) cho hệ số rất gần lý thuyết. Một hệ số chỉ hơn 1 một chút, lặp lại qua 29 lớp, thành
hàng trăm lần. Yang et al. còn chỉ ra rằng chính BatchNorm gây ra điều này: đổi khởi tạo hay hàm kích hoạt không
chữa được.

Gradient lớn ở lớp đầu thì sao? Mọi lớp dùng chung một $\eta$. Nếu $\eta$ đủ nhỏ để lớp đầu bước vừa phải, các lớp
cuối gần như đứng yên; nếu $\eta$ đủ lớn cho các lớp cuối, bước đầu tiên đã phá trọng số của lớp đầu. Ô dưới chạy
**đúng một bước** SGD với $\eta = 0{,}03$ trên mạng 30 lớp và đo mỗi lớp thay đổi bao nhiêu, rồi theo dõi tỉ số
gradient trong 100 bước đầu:

In [ ]:
for norm in ("none", "bn"):
    P0, rec = init(norm, 30), {}

    def watch(step, P, G, rec=rec, P0=P0):
        if G is not None:
            rec[step] = float(np.linalg.norm(G["W"][0]) / np.linalg.norm(G["W"][29]))  # gradient trước bước `step`
        if step == 1:
            rec["move"] = [np.linalg.norm(P["W"][l] - P0["W"][l]) / np.linalg.norm(P0["W"][l]) for l in (0, 14, 29)]
            rec["cos"] = float((P["W"][0] * P0["W"][0]).sum() / np.linalg.norm(P["W"][0]) / np.linalg.norm(P0["W"][0]))
            rec["len"] = float(np.linalg.norm(P["W"][0]) / np.linalg.norm(P0["W"][0]))

    train(norm, 30, 0.03, steps=101, watch=watch)
    m = rec["move"]
    print(
        f"{LABEL[norm]}, sau 1 bước: ‖ΔW‖/‖W‖ ở lớp 1 · 15 · 30 = {pct(m[0], 2)} · {pct(m[1], 2)} · {pct(m[2], 2)}; ",
        end="",
    )
    print(f"W₁ dài gấp {vn(rec['len'], 2)} và quay đi {vn(math.degrees(math.acos(min(1.0, rec['cos']))), 0)}°")
    print(f"   ‖∂L/∂W₁‖ / ‖∂L/∂W₃₀‖ ở bước 1 · 2 · 11 · 101: {' · '.join(vn(rec[t], 1) for t in (1, 2, 11, 101))}")

Ở mạng không chuẩn hóa, một bước chỉ dịch mỗi lớp đi một phần rất nhỏ. Ở mạng BatchNorm, cùng $\eta = 0{,}03$ đó,
bước đầu tiên thay $W_1$ bằng một ma trận khác hẳn: $\|\Delta W_1\|$ lớn hơn cả $W_1$, và $W_1$ dài gấp đôi, quay đi
khoảng 60°. Những gì khởi tạo He chuẩn bị cho lớp đầu bị xóa ngay ở bước 1, trong khi lớp 30 gần như không nhúc
nhích. Sau đó tỉ số gradient giảm nhanh: $W_1$ dài ra thì gradient của nó nhỏ lại (sự bất biến tỉ lệ ở mục 9), và
Yang et al. (2019) cũng thấy rằng sau một bước tối ưu, mạng đạt một trạng thái cân bằng tương đối ổn định.

Mạng BatchNorm 30 lớp vì thế kẹt giữa hai lựa chọn. $\eta$ nhỏ thì các lớp cuối học rất chậm, mà lớp đầu vẫn bị xáo
trộn ở bước 1 (ở $\eta = 0{,}01$ nó chỉ đạt 48–51%). $\eta$ lớn hơn thì xáo trộn mạnh hơn và các lớp đầu bị giảm tốc
độ học hiệu dụng. Đây là cơ chế mà Yang et al. mô tả; notebook đo được nó nhưng không chứng minh nó là lý do duy nhất.
Bài học rút ra: **chuẩn hóa thôi chưa đủ để huấn luyện mạng rất sâu.** Mục 13 cho thấy lời giải của Bài 13.

## 11 · LayerNorm, RMSNorm và Transformer

LayerNorm chuẩn hóa từng mẫu bằng thống kê của riêng nó, nên kết quả của một ảnh không phụ thuộc lô, và khi huấn
luyện hay suy luận nó làm cùng một phép tính. Ô dưới kiểm tra điều đó trên 10.000 đầu vào thật của lớp 5 (mạng của
mục 7). Nó cũng kiểm tra một nhận xét của Zhang & Sennrich (2019): khi trung bình của mỗi hàng bằng 0, RMSNorm
trùng khít LayerNorm.

In [ ]:
ln_all = norm_forward(A_all, "ln", one, zero)[0]
ln_one = norm_forward(A_all[:1], "ln", one, zero)[0]  # ảnh 1 đứng một mình
bn_8, bn_64 = norm_forward(A_all[:8], "bn", one, zero)[0], norm_forward(A_all[:64], "bn", one, zero)[0]
print(f"LayerNorm của ảnh 1, một mình hay trong 10.000 ảnh: sai khác {sci(np.abs(ln_one[0] - ln_all[0]).max())}")
print(f"BatchNorm của ảnh 1, trong lô 8 hay lô 64 ảnh: sai khác tới {vn(np.abs(bn_8[0] - bn_64[0]).max(), 2)}")
centered = A_all - A_all.mean(1, keepdims=True)  # mỗi hàng có trung bình 0
gap = np.abs(norm_forward(centered, "rms", one, zero)[0] - norm_forward(centered, "ln", one, zero)[0]).max()
print(f"hàng có trung bình 0: RMSNorm − LayerNorm = {sci(gap)}")
row_mu = np.abs(A_all.mean(1))
print(f"nhưng a thật có |trung bình hàng| từ {vn(row_mu.min(), 2)} đến {vn(row_mu.max(), 2)}, nên hai cách khác nhau")
assert np.abs(ln_one[0] - ln_all[0]).max() < 1e-12 and gap < 1e-12

LayerNorm của một ảnh y hệt nhau dù ảnh đó đứng một mình hay nằm giữa 10.000 ảnh, còn BatchNorm của cùng ảnh đó đổi
hẳn khi lô đổi từ 8 sang 64 ảnh. Trong Transformer, mỗi **token** là một hàng: LayerNorm chuẩn hóa vectơ của từng
token trên các chiều của nó, không cần token hay câu nào khác. Vì vậy nó hợp với chuỗi dài ngắn khác nhau và với lô
rất nhỏ. Cách đặt nó trong mô hình cũng đã thay đổi:

- **Transformer gốc** (Vaswani et al., 2017) đặt LayerNorm *sau* mỗi lớp con: $\mathrm{LN}(x + \mathrm{Sublayer}(x))$
  (post-norm).
- **GPT-2** (Radford et al., 2019) chuyển LayerNorm lên *trước* mỗi lớp con, $x + \mathrm{Sublayer}(\mathrm{LN}(x))$
  (pre-norm), và thêm một LayerNorm ở cuối.
- **LLaMA** (Touvron et al., 2023, mục 2.2) giữ pre-norm và đổi sang RMSNorm. Các mô hình mở của Qwen và DeepSeek cũng
  dùng RMSNorm (tệp cấu hình công khai của chúng có tham số `rms_norm_eps`).

Zhang & Sennrich (2019) giả thuyết rằng phần quan trọng của LayerNorm là chia cho độ lớn, còn trừ trung bình thì không
cần. Trên các mô hình họ thử, RMSNorm đạt chất lượng tương đương LayerNorm mà giảm thời gian chạy 7–64%. Bài 17
(Transformer & GPT) quay lại pre-norm bên trong một khối Transformer.

Trong mạng tích chập, BatchNorm vẫn phổ biến: mỗi kênh được chuẩn hóa trên cả lô và mọi vị trí trong ảnh (Ioffe &
Szegedy, 2015, mục 3.2). Nhưng ConvNeXt (Liu et al., 2022) cho thấy một mạng tích chập hiện đại dùng LayerNorm thay
cho BatchNorm vẫn huấn luyện tốt, và còn tốt hơn một chút (81,5% top-1 trên ImageNet).

## 12 · Thử thay đổi

Các ô dưới chạy lại cùng các hàm với tham số khác. Bạn có thể sửa giá trị rồi chạy lại.

### Nhiễu sau BatchNorm, trên ba seed
Mục 6 chỉ thử một seed. Ô dưới chạy thêm seed 2 và 3 (mạng 10 lớp, $\eta = 0{,}3$), và so với BatchNorm sạch và mạng
không chuẩn hóa ($\eta = 0{,}03$) của cùng seed, lấy từ tệp của demo. Đổi `LEVEL` để thử nhiễu mạnh hơn (0,5) hoặc
`ETA_SHAKE` để thử tốc độ học khác.

In [ ]:
LEVEL, ETA_SHAKE = 0.3, 0.3
shaken = {1: val_shake} if (LEVEL, ETA_SHAKE) == (0.3, 0.3) else {}  # seed 1 đã chạy ở mục 6
for seed in (1, 2, 3):
    if seed not in shaken:
        P, _ = train("bn", 10, ETA_SHAKE, seed=seed, shake=LEVEL)
        shaken[seed] = accuracy(P, Xval, yval)
for seed in (1, 2, 3):
    clean, plain = stored("bn", 10, ETA_SHAKE, seed), stored("none", 10, 0.03, seed)
    print(f"seed {seed}: BatchNorm có nhiễu {pct(shaken[seed])} · BatchNorm sạch {show(clean)} · ", end="")
    print(f"không chuẩn hóa {show(plain)}")
clean3 = [stored("bn", 10, ETA_SHAKE, s)["val"] for s in (1, 2, 3)]
plain3 = [stored("none", 10, 0.03, s)["val"] for s in (1, 2, 3)]
for name, v in [("có nhiễu", list(shaken.values())), ("sạch", clean3), ("không chuẩn hóa", plain3)]:
    print(f"  {name:>16}: {pct(min(v))}–{pct(max(v))} (3 seed)")
print(f"BatchNorm sạch, seed 3, η = 0,1: {show(stored('bn', 10, 0.1, 3))}")

Trên 3 seed, khoảng của BatchNorm có nhiễu nằm gọn trong khoảng của BatchNorm sạch: khác biệt nằm **trong dao động
giữa các seed**. Seed 3 sạch chỉ đạt 87,02%: ở seed đó $\eta = 0{,}3$ đã là quá lớn (cùng seed, $\eta = 0{,}1$ đạt
95,96%), và nhiễu lại cho kết quả tốt hơn. Kết luận của mục 6 vẫn đứng: mạng chịu được một mức trôi dạt lớn hơn nhiều
lần mức trôi tự nhiên. Thử `LEVEL = 0.5` để xem nhiễu mạnh đến đâu thì bắt đầu có hại.

### Kích thước lô và tốc độ bùng nổ gradient
Công thức của Yang et al. (2019) phụ thuộc vào số ảnh $B$ trong lô: lô càng nhỏ, gradient càng tăng nhanh qua mỗi lớp.
Ô dưới đo hệ số đó trên mạng BatchNorm 30 lớp lúc khởi tạo, với $B$ ảnh đầu của thứ tự seed 1. Đổi `SIZES_B` để thử.

In [ ]:
SIZES_B = [8, 16, 64, 256]
order1 = np.random.default_rng(1).permutation(N_TRAIN)
for B in SIZES_B:
    g = layer_grads("bn", 30, Xtr[order1[:B]], ytr[order1[:B]])
    f = float(np.exp(-np.polyfit(k, np.log(g[1:]), 1)[0]))
    print(f"lô {B:>3} ảnh: gradient nhân {vn(f, 3)} mỗi lớp (lý thuyết {vn(math.sqrt(yang_rate(B)), 3)}); ", end="")
    print(f"‖∂L/∂W₁‖ / ‖∂L/∂W₃₀‖ = {vn(g[0] / g[-1], 0)}")

Đo được luôn thấp hơn lý thuyết một chút (lý thuyết dành cho mạng rộng vô hạn, mạng này rộng 64), nhưng xu hướng
trùng: lô nhỏ thì gradient bùng nổ nhanh hơn. Đây là thêm một lý do để tránh BatchNorm với lô rất nhỏ. Wu & He (2018)
đề xuất GroupNorm, chuẩn hóa theo nhóm kênh của từng mẫu, cho đúng trường hợp đó.

## 13 · Giới hạn và cầu nối sang bài sau

Chuẩn hóa giữ thang đo của từng lớp ngay ngắn và cho phép dùng $\eta$ lớn hơn. Nhưng trong một mạng *thẳng*, mọi tín
hiệu và mọi gradient vẫn phải đi qua đủ 30 lớp, và với BatchNorm mỗi lớp lại nhân gradient thêm khoảng 1,2 lần. Bài 13
thêm **kết nối tắt** (residual): mỗi lớp chỉ *cộng thêm* một phần vào $h$ thay vì thay thế nó,

$$h \leftarrow h + \mathrm{ReLU}(\mathrm{BN}(h))\,W$$

(dạng “pre-activation” của He et al., 2016; lớp 784 → 64 đầu tiên vẫn là lớp thẳng). Gradient vì thế có một đường đi
thẳng từ lớp ra về lớp đầu. Ô dưới dựng mạng đó với **đúng các ma trận $W$ của mạng thẳng ở mục 10** và đo lại tỉ số
gradient lúc khởi tạo, trên cùng lô đầu tiên của seed 1.

In [ ]:
def residual_grads(depth, bn=True, x=x1, y=y1):
    """‖∂L/∂W_l‖ lúc khởi tạo của mạng có kết nối tắt h ← h + ReLU(N(h))·W (Bài 13), cùng W với mạng thẳng."""
    W = init("none", depth)["W"]

    def N(a):  # BatchNorm (γ = 1, β = 0) hoặc không gì cả
        return norm_forward(a, "bn", one, zero) if bn else (a, None)

    def N_back(dy, c):
        return norm_backward(dy, "bn", c, one)[0] if bn else dy

    y0, c0 = N(x @ W[0])  # lớp 1: 784 → 64, thẳng
    h, cache = np.maximum(0, y0), []
    for l in range(1, depth):
        y_l, c_l = N(h)
        r = np.maximum(0, y_l)
        cache.append((r, y_l, c_l))
        h = h + r @ W[l]  # kết nối tắt
    y_out, c_out = N(h)
    z = np.maximum(0, y_out) @ W[-1]
    dz = softmax(z)
    dz[np.arange(len(y)), y] -= 1
    dh = N_back((dz / len(y) @ W[-1].T) * (y_out > 0), c_out)
    g = [0.0] * depth
    for l in range(depth - 1, 0, -1):
        r, y_l, c_l = cache[l - 1]
        g[l] = np.linalg.norm(r.T @ dh)
        dh = dh + N_back((dh @ W[l].T) * (y_l > 0), c_l)  # gradient đi qua đường tắt và qua lớp
    g[0] = np.linalg.norm(x.T @ N_back(dh * (y0 > 0), c0))
    return np.array(g)


ratio = {}
for depth in sweep["depths"]:
    plain, res = layer_grads("bn", depth), residual_grads(depth)
    ratio[depth] = (plain[0] / plain[-1], res[0] / res[-1])
    print(f"{depth} lớp, BatchNorm: ‖∂L/∂W₁‖ / ‖∂L/∂W_L‖ = {vn(ratio[depth][0], 1)} (thẳng) → ", end="")
    print(f"{vn(ratio[depth][1], 1)} (có kết nối tắt)")
print(f"từ 10 lên 30 lớp, tỉ số tăng {vn(ratio[30][0] / ratio[10][0], 0)} lần (thẳng), ", end="")
print(f"{vn(ratio[30][1] / ratio[10][1], 1)} lần (có kết nối tắt)")
assert abs(ratio[30][1] - 38.56) < 0.05  # đúng như reel/mnist_mlp/resmlp.py, dữ liệu của Bài 13

Ở 10 lớp, hai mạng ngang nhau. Nhưng khi đi từ 10 lên 30 lớp, tỉ số của mạng thẳng tăng 57 lần (theo cấp số nhân),
còn của mạng có kết nối tắt chỉ tăng khoảng 2 lần. Cùng các ma trận $W$, cùng BatchNorm, chỉ khác ở phép cộng
$h + \dots$. Bài 13 sẽ huấn luyện mạng này tới 100 lớp và xem nó có thật sự học tốt hơn không.

Bài tiếp theo (Bài 11) chuyển sang ảnh: mạng tích chập. BatchNorm sẽ quay lại ở đó dưới dạng mỗi **kênh** một cặp
$\mu, \sigma$, tính trên cả lô và mọi vị trí trong ảnh.

## Tóm tắt
- Ba phép chuẩn hóa cùng công thức $\hat x = (x - \mu)/\sqrt{\sigma^2 + \epsilon}$, $y = \gamma\hat x + \beta$, chỉ
  khác trục: BatchNorm theo cột (mỗi nơ-ron trên lô), LayerNorm theo hàng (mỗi mẫu trên các nơ-ron), RMSNorm theo
  hàng nhưng không trừ $\mu$ và không có $\beta$. Chiều ngược của cả ba khớp với sai phân hữu hạn tới cỡ $10^{-10}$.
- Không chuẩn hóa, đầu vào của lớp 5 trôi đi trong 1 epoch (độ lệch chuẩn tăng nhiều lần); BatchNorm giữ đầu ra của nó
  ở trung bình gần 0, độ lệch chuẩn gần 1. Nhưng cố tình thêm nhiễu mạnh sau BatchNorm chỉ làm mạng kém đi trong dao
  động giữa các seed: chặn trôi dạt không phải toàn bộ lý do BatchNorm giúp (Santurkar et al., 2018).
- Với BatchNorm, $\gamma = \sqrt{\sigma^2+\epsilon}$, $\beta = \mu$ trả lại đúng $a$; độ lệch $b$ trước nó không bao
  giờ nhận gradient.
- Suy luận phải dùng trung bình trượt: mạng đúng 95,32%, còn nếu dùng thống kê của lô thì 93,88% (lô 64), 75,47%
  (lô 8) và 10,32% (lô 1, mọi ảnh cho cùng một đáp án).
- BatchNorm không đổi khi nhân $W$ với một số, còn gradient nhỏ đi đúng số lần đó, nên nó tự hãm các bước quá lớn:
  với $\eta = 1$ mạng 10 lớp vẫn đạt 96,04%, trong khi mạng không chuẩn hóa nổ và LayerNorm, RMSNorm sụp về đoán mò.
- Ở 10 và 20 lớp BatchNorm hơn rõ; ở 30 lớp không cách nào hơn rõ mạng không chuẩn hóa. Mạng BatchNorm thẳng có gradient
  tăng khoảng 1,21 lần mỗi lớp khi đi ngược (Yang et al., 2019): ‖∂L/∂W₁‖ gấp 899 lần ‖∂L/∂W₃₀‖, và bước SGD đầu tiên
  thay $W_1$ bằng một ma trận khác hẳn. Kết nối tắt (Bài 13) hạ tỉ số đó xuống 38,6.

## Câu hỏi ôn tập
1. Với ma trận kích hoạt $a$ (hàng = ảnh, cột = nơ-ron), BatchNorm và LayerNorm tính $\mu, \sigma^2$ theo trục nào?
   Cách nào cho kết quả của một ảnh phụ thuộc vào các ảnh khác?
2. Vì sao độ lệch $b$ của lớp Linear ngay trước BatchNorm không bao giờ nhận gradient? Cái gì làm thay việc của nó?
3. Một mạng BatchNorm đúng 95,32% khi suy luận đúng cách. Vì sao đưa từng ảnh một qua mạng mà vẫn dùng thống kê của
   lô thì chỉ còn đúng 10,32%, và vì sao lại đúng con số đó?
4. Thí nghiệm “BatchNorm có nhiễu” ở mục 6 nói gì về giả thuyết dịch chuyển hiệp biến nội tại?
5. Nhân $W$ của một lớp BatchNorm với 3 thì đầu ra và gradient theo $W$ thay đổi ra sao? Điều đó giúp gì khi $\eta$ lớn?
6. Vì sao BatchNorm giúp mạng 10 lớp mà không giúp mạng 30 lớp thẳng? Kết nối tắt thay đổi điều gì?

<details><summary>Gợi ý đáp án</summary>

1. BatchNorm theo cột (mỗi nơ-ron, trên các ảnh của lô), LayerNorm theo hàng (mỗi ảnh, trên các nơ-ron của nó).
   BatchNorm làm kết quả của một ảnh phụ thuộc các ảnh cùng lô; LayerNorm thì không, và nó tính như nhau khi huấn
   luyện và khi suy luận.
2. BatchNorm trừ trung bình của cột, nên cộng thêm một hằng số $b$ vào cả cột không làm thay đổi gì; trong chiều ngược,
   $\partial L/\partial a$ cộng theo cột luôn bằng 0. $\beta$ làm thay việc của $b$.
3. Với một ảnh, $\mu$ của mỗi nơ-ron chính là giá trị của nó, nên $x - \mu = 0$: ảnh biến mất ngay ở lớp đầu, mọi ảnh
   cho cùng một logit, và mạng luôn đoán cùng một chữ số (số 2). Độ chính xác bằng tỉ lệ số 2 trong tập kiểm tra,
   1.032 / 10.000.
4. Nhiễu đổi trung bình và thang đo của mọi nơ-ron ở mọi bước, mạnh hơn nhiều lần mức trôi tự nhiên, mà mạng vẫn học
   gần tốt như BatchNorm sạch. Vậy chặn trôi dạt không thể là toàn bộ lý do BatchNorm giúp.
5. Đầu ra gần như không đổi, còn gradient nhỏ đi đúng 3 lần. Bước lớn làm $W$ dài ra, và $W$ dài thì bước tương đối
   $\|\Delta W\|/\|W\|$ nhỏ lại: BatchNorm tự giảm tốc độ học hiệu dụng, nên không nổ.
6. Trong mạng thẳng, BatchNorm nhân gradient thêm khoảng 1,2 lần mỗi lớp khi đi ngược, nên ở 30 lớp gradient của
   $W_1$ gấp 899 lần của $W_{30}$, và bước đầu tiên đã xáo trộn lớp đầu. Kết nối tắt cho gradient một đường đi thẳng về
   lớp đầu; với cùng các $W$, tỉ số còn 38,6.
</details>

## Tìm hiểu thêm
- S. Ioffe & C. Szegedy (2015), [Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate
  Shift](https://arxiv.org/abs/1502.03167): mục 3.1 (suy luận bằng thống kê của tập huấn luyện), 3.2 (bỏ $b$; mạng
  tích chập), 3.3 (bất biến tỉ lệ).
- J. L. Ba, J. R. Kiros & G. E. Hinton (2016), [Layer Normalization](https://arxiv.org/abs/1607.06450).
- B. Zhang & R. Sennrich (2019), [Root Mean Square Layer Normalization](https://arxiv.org/abs/1910.07467).
- S. Santurkar, D. Tsipras, A. Ilyas & A. Madry (2018), [How Does Batch Normalization Help
  Optimization?](https://arxiv.org/abs/1805.11604): thí nghiệm “BatchNorm có nhiễu” và địa hình mất mát trơn hơn.
- J. Bjorck, C. Gomes, B. Selman & K. Q. Weinberger (2018), [Understanding Batch
  Normalization](https://arxiv.org/abs/1806.02375): BatchNorm giúp chủ yếu vì cho phép $\eta$ lớn.
- S. Arora, Z. Li & K. Lyu (2019), [Theoretical Analysis of Auto Rate-Tuning by Batch
  Normalization](https://arxiv.org/abs/1812.03981).
- G. Yang, J. Pennington, V. Rao, J. Sohl-Dickstein & S. S. Schoenholz (2019), [A Mean Field Theory of Batch
  Normalization](https://arxiv.org/abs/1902.08129): Định lý 3.10, gradient bùng nổ trong mạng BatchNorm thẳng.
- Y. Wu & J. Johnson (2021), [Rethinking “Batch” in BatchNorm](https://arxiv.org/abs/2105.07576): PreciseBN và các
  cạm bẫy của BatchNorm.
- Y. Wu & K. He (2018), [Group Normalization](https://arxiv.org/abs/1803.08494).
- Z. Liu et al. (2022), [A ConvNet for the 2020s](https://arxiv.org/abs/2201.03545), mục 2.6: thay BatchNorm bằng
  LayerNorm trong ConvNeXt.
- H. Touvron et al. (2023), [LLaMA](https://arxiv.org/abs/2302.13971), mục 2.2: pre-norm với RMSNorm.
- A. Karpathy, [Neural Networks: Zero to Hero](https://karpathy.ai/zero-to-hero.html), bài “Building makemore Part 3”:
  BatchNorm viết tay trong PyTorch, cùng các biểu đồ kích hoạt và gradient.

**Bài tiếp theo:** [Bài 11 · Mạng tích chập (CNN)](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/cnn/): thay lớp kết
nối đầy đủ bằng các bộ lọc nhỏ trượt trên ảnh, để mạng dùng lại cùng một trọng số ở mọi vị trí.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")